# 48 · Ensemble **epoch** và **seed** — miễn phí, dữ liệu đã nằm trên đĩa

`38c`/`38d`/`44b`/`45` đã `np.save` dự đoán val cho **mọi arm, mọi epoch, mọi seed**. Chưa ai chạm.

`17` đo trên cohort cũ, ba checkpoint **cùng một lần chạy**:

```
macro  +0.00112  CI [+0.00027, +0.00197]  loai 0
worst3 +0.00345  CI [+0.00035, +0.00645]  loai 0
tail   +0.00465  CI [-0.00230, +0.01213]  chua 0
```

## Vì sao kết quả âm của `35 §6` và `47` KHÔNG áp vào đây

`35 §6` thấy ensemble trọng số đều **âm** — nhưng nguyên nhân ghi rõ: thành viên **lệch chất lượng**
(`cls 0.1651` vs `patch 0.2028`). Các epoch liền nhau của cùng một lần chạy thì **ngang nhau**.

`47` thấy trộn **theo nhãn** thua — nhưng đó là chọn **nguồn cho từng nhãn**, cơ chế khác hẳn với
**trung bình nhiều mô hình** trên mọi nhãn.

## Quy tắc CỐ ĐỊNH, không chọn trên val

Chọn "các epoch quanh đỉnh" bằng val rồi báo cáo trên val là rò rỉ. Nên dùng quy tắc **khai báo
trước**, không nhìn dữ liệu:

```
last-2 · last-3 · last-5 · all      <- co dinh, khong chon gi
seed-mean                            <- trung binh cac seed o EPOCH CUOI
epoch x seed                         <- ca hai
cross-fit                            <- chon epoch o nua nay, cham o nua kia (de doi chieu)
```

## Trung bình **HẠNG** song song trung bình xác suất

`AP` là chỉ số **xếp hạng**. Các epoch có calibration khác nhau (LR khác, nhất là epoch cuối đã ủ),
nên trung bình xác suất có thể bị epoch tự tin nhất chi phối. Trung bình hạng miễn nhiễm.

## C0 · Dò các lần chạy đã có, không hard-code

In [1]:
import os, json, re, glob
import numpy as np, pandas as pd

try:
    from google.colab import drive; drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = './planB_out'
OUT_DIR = os.path.join(DATA_DIR, 'cxrlt_2024'); EMB_DIR = os.path.join(OUT_DIR, 'embeddings')
RES48 = os.path.join(OUT_DIR, 'stage4_nb48')
def ensure_res():
    os.makedirs(RES48, exist_ok=True); return RES48
ensure_res(); open(os.path.join(RES48, '_run_info.json'), 'w').write('{"notebook":"48"}')

INV = json.load(open(os.path.join(OUT_DIR, 'cxrlt_inventory.json')))
LAB40 = INV['labels_task1']; C = len(LAB40)
TIER = {t: np.array([LAB40.index(l) for l in INV['tier'][t]]) for t in ('head','medium','tail')}
ORDER = pd.read_parquet(os.path.join(EMB_DIR, 'emb_index.parquet'))
Y = (pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_index.parquet'), columns=['dicom_id']+LAB40)
       .set_index('dicom_id').loc[ORDER.dicom_id].reset_index())
YM = Y[LAB40].to_numpy('float32')
VA = ORDER.split.to_numpy() == 'val'
SUBV = ORDER.subject_id.to_numpy()[VA]; YV = YM[VA]; N_VA = int(VA.sum())

# --- do file: p_{arm}_s{seed}_ep{k}.npy ---
PAT = re.compile(r'^p_(?P<arm>.+)_s(?P<seed>\d+)_ep(?P<ep>\d+)\.npy$')
FILES = {}
for d in sorted(glob.glob(os.path.join(OUT_DIR, 'stage4_nb*', '*'))):
    if not os.path.isdir(d): continue
    run = f'{os.path.basename(os.path.dirname(d))[9:]}/{os.path.basename(d)}'
    for f in sorted(os.listdir(d)):
        m = PAT.match(f)
        if m: FILES.setdefault((run, m['arm']), {}).setdefault(int(m['seed']), []).append(
                  (int(m['ep']), os.path.join(d, f)))
for k in FILES:
    for s in FILES[k]: FILES[k][s].sort()
print(f'{len(FILES)} cap (lan chay, arm) co du doan da luu:')
for (run, arm), sd in sorted(FILES.items()):
    print(f'  {run:<28} {arm:<16} seed {sorted(sd)} | epoch {[e for e,_ in sd[sorted(sd)[0]]]}')

_u = np.unique(SUBV); _rs = np.random.default_rng(0); _rs.shuffle(_u)
FA = np.isin(SUBV, _u[:len(_u)//2]); FB = ~FA           # cat THEO BENH NHAN
assert not (set(SUBV[FA]) & set(SUBV[FB]))
print(f'\nval {N_VA:,} anh | nua A {FA.sum():,} / nua B {FB.sum():,} | khong chung benh nhan')

Mounted at /content/drive
55 cap (lan chay, arm) co du doan da luu:
  38b/b1_pre_vs_ft             ft_mean_lin      seed [42, 43] | epoch [1, 2, 3, 4, 5, 6, 7, 8]
  38b/b1_pre_vs_ft             ft_mean_lin_ehr  seed [42, 43] | epoch [1, 2, 3, 4, 5, 6, 7, 8]
  38b/b1_pre_vs_ft             ft_mldec         seed [42, 43] | epoch [1, 2, 3, 4, 5, 6, 7, 8]
  38b/b1_pre_vs_ft             ft_mldec_cat     seed [42, 43] | epoch [1, 2, 3, 4, 5, 6, 7, 8]
  38b/b1_pre_vs_ft             ft_mldec_col_cat seed [42, 43] | epoch [1, 2, 3, 4, 5, 6, 7, 8]
  38b/b1_pre_vs_ft             ft_mldec_col_gate seed [42, 43] | epoch [1, 2, 3, 4, 5, 6, 7, 8]
  38b/b1_pre_vs_ft             ft_mldec_gate    seed [42, 43] | epoch [1, 2, 3, 4, 5, 6, 7, 8]
  38b/b1_pre_vs_ft             ft_mldec_gate_k1 seed [42, 43] | epoch [1, 2, 3, 4, 5, 6, 7, 8]
  38b/b1_pre_vs_ft             ft_mldec_gate_zero seed [42, 43] | epoch [1, 2, 3, 4, 5, 6, 7, 8]
  38b/b1_pre_vs_ft             ft_mldec_t1      seed [42, 43] | epoch [1, 

## C1 · Chấm điểm + hai phép trung bình

In [2]:
def ap_cols(Yc, Sc):
    o = np.full(Yc.shape[1], np.nan)
    for j in range(Yc.shape[1]):
        y, sc = Yc[:, j], Sc[:, j]; n1 = y.sum()
        if n1 == 0 or n1 == len(y): continue
        r = np.argsort(-sc, kind='stable'); yy, ss = y[r], sc[r]
        tp, fp = np.cumsum(yy), np.cumsum(1-yy)
        last = np.r_[np.diff(ss) != 0, True]; tp, fp = tp[last], fp[last]
        pr, rc = tp/(tp+fp), tp/n1
        o[j] = float((np.diff(np.r_[0.0, rc])*pr).sum())
    return o
tiers = lambda ap: {t: float(np.nanmean(ap[i])) for t, i in TIER.items()}
sc = lambda P: dict(mAP=float(np.nanmean(ap_cols(YV, P))), **tiers(ap_cols(YV, P)))

def rank_mean(Ps):
    '''Trung binh HANG. AP la chi so xep hang, ma cac epoch co calibration khac nhau (epoch cuoi
       da u LR ve 0) nen trung binh XAC SUAT co the bi epoch tu tin nhat chi phoi.'''
    R = np.zeros_like(Ps[0])
    for P in Ps:
        R += np.argsort(np.argsort(P, 0), 0).astype('float32')
    return R / (len(Ps) * len(R))
prob_mean = lambda Ps: np.mean(Ps, 0)
print('san sang')

san sang


## C2 · Quy tắc **cố định** — không nhìn dữ liệu để chọn

`base` là epoch cuối, đúng giao thức `nb35 §0.1`. Mọi dòng khác so với nó.

In [3]:
rows = []
for (run, arm), sd in sorted(FILES.items()):
    seeds = sorted(sd)
    ld = lambda p: np.load(p).astype('float32')
    for s in seeds:
        eps = sd[s]                                  # [(ep, path), ...] da sap
        if len(eps) < 2: continue
        base = ld(eps[-1][1])                        # EPOCH CUOI
        rows.append(dict(run=run, arm=arm, seed=str(s), cach='base (epoch cuoi)', n=1, **sc(base)))
        for k in (2, 3, 5, len(eps)):
            if k > len(eps) or k < 2: continue
            Ps = [ld(p) for _, p in eps[-k:]]
            nm = 'all' if k == len(eps) else f'last-{k}'
            rows.append(dict(run=run, arm=arm, seed=str(s), cach=f'{nm} prob', n=k, **sc(prob_mean(Ps))))
            rows.append(dict(run=run, arm=arm, seed=str(s), cach=f'{nm} rank', n=k, **sc(rank_mean(Ps))))
    if len(seeds) > 1:                                # ENSEMBLE SEED o epoch cuoi
        Ps = [ld(sd[s][-1][1]) for s in seeds]
        rows.append(dict(run=run, arm=arm, seed='TB', cach='seed-mean prob', n=len(seeds), **sc(prob_mean(Ps))))
        rows.append(dict(run=run, arm=arm, seed='TB', cach='seed-mean rank', n=len(seeds), **sc(rank_mean(Ps))))
        Ps = [ld(p) for s in seeds for _, p in sd[s][-3:]]      # epoch x seed
        rows.append(dict(run=run, arm=arm, seed='TB', cach='last-3 x seed prob', n=len(Ps), **sc(prob_mean(Ps))))
        rows.append(dict(run=run, arm=arm, seed='TB', cach='last-3 x seed rank', n=len(Ps), **sc(rank_mean(Ps))))
T = pd.DataFrame(rows)
ensure_res(); T.to_csv(os.path.join(RES48, 'ensemble.csv'), index=False)

print('=== HIEU so voi `base` (epoch cuoi), TB qua cac (lan chay, arm, seed) ===')
piv = T.pivot_table(index='cach', values=['mAP','head','medium','tail'], aggfunc='mean')
b = T[T.cach == 'base (epoch cuoi)']
key = ['run','arm','seed']
D = (T.merge(b[key+['mAP','tail']], on=key, suffixes=('','_b'))
       .assign(d_mAP=lambda x: x['mAP']-x['mAP_b'],       # x.tail la PHUONG THUC DataFrame.tail
               d_tail=lambda x: x['tail']-x['tail_b']))   # -> luon dung ngoac vuong
g = D[D.cach != 'base (epoch cuoi)'].groupby('cach')
print(pd.DataFrame(dict(n_cap=g.size(), d_mAP=g.d_mAP.mean(), thang=g.d_mAP.apply(lambda v: (v>0).mean()),
                        d_tail=g.d_tail.mean())).to_string(float_format=lambda v: f'{v:+.4f}'))
print('\n  `thang` = ty le (lan chay, arm, seed) ma cach do HON epoch cuoi.')
print('  Khong co bien do seed o day -> doc `thang` (co bao nhieu cap cung dau) chu dung doc do lon.')

=== HIEU so voi `base` (epoch cuoi), TB qua cac (lan chay, arm, seed) ===
             n_cap   d_mAP   thang  d_tail
cach                                      
all prob        70 +0.0103 +0.8857 +0.0086
all rank        70 +0.0052 +0.7714 +0.0016
last-2 prob     70 +0.0003 +0.8286 +0.0002
last-2 rank     70 +0.0001 +0.5857 -0.0002
last-3 prob     70 +0.0014 +0.9143 +0.0011
last-3 rank     70 +0.0013 +0.8857 +0.0008
last-5 prob     70 +0.0040 +0.8857 +0.0041
last-5 rank     70 +0.0038 +0.8857 +0.0037

  `thang` = ty le (lan chay, arm, seed) ma cach do HON epoch cuoi.
  Khong co bien do seed o day -> doc `thang` (co bao nhieu cap cung dau) chu dung doc do lon.


## C3 · Bảng cho các arm mạnh nhất

In [4]:
TOP = ['t1_prior', 'mldec_t1', 'mldec_col_cat', 'mldec']
for (run, arm), _ in sorted(FILES.items()):
    if arm not in TOP: continue
    sub = T[(T.run == run) & (T.arm == arm)]
    if sub.empty: continue
    print(f'=== {run} · {arm} ===')
    print(sub[['seed','cach','n','mAP','head','medium','tail']].to_string(
        index=False, float_format=lambda v: f'{v:.4f}'))
    print('')

=== 38c/c1_micro_full · mldec ===
seed               cach  n    mAP   head  medium   tail
  42  base (epoch cuoi)  1 0.2419 0.4545  0.1720 0.0925
  42        last-2 prob  2 0.2421 0.4549  0.1717 0.0932
  42        last-2 rank  2 0.2419 0.4548  0.1717 0.0925
  42        last-3 prob  3 0.2433 0.4559  0.1732 0.0941
  42        last-3 rank  3 0.2431 0.4558  0.1731 0.0937
  42        last-5 prob  5 0.2473 0.4583  0.1772 0.0999
  42        last-5 rank  5 0.2470 0.4584  0.1768 0.0992
  42           all prob  8 0.2505 0.4608  0.1816 0.1022
  42           all rank  8 0.2493 0.4609  0.1815 0.0978
  43  base (epoch cuoi)  1 0.2447 0.4557  0.1717 0.1016
  43        last-2 prob  2 0.2448 0.4559  0.1718 0.1013
  43        last-2 rank  2 0.2447 0.4559  0.1717 0.1012
  43        last-3 prob  3 0.2454 0.4566  0.1732 0.1008
  43        last-3 rank  3 0.2453 0.4566  0.1732 0.1005
  43        last-5 prob  5 0.2477 0.4587  0.1762 0.1022
  43        last-5 rank  5 0.2480 0.4588  0.1763 0.1031
  43          

## C4 · Ghi `REVIEW_ENSEMBLE.md`

In [5]:
L = []; w = L.append
w('# REVIEW · ensemble epoch & seed (`nb48`, hậu kỳ, không huấn luyện lại)'); w('')
w('Dự đoán val đã lưu của `38c`/`38d`/`44b`/`45`. Quy tắc **cố định** (`last-k`, `all`, `seed-mean`) '
  '— không chọn epoch bằng val. `base` = **epoch cuối**, đúng giao thức `nb35 §0.1`.'); w('')
w('| cách | #cặp | Δ mAP | tỉ lệ thắng | Δ tail |'); w('|---|---|---|---|---|')
for c, r in pd.DataFrame(dict(n_cap=g.size(), d_mAP=g.d_mAP.mean(),
                              thang=g.d_mAP.apply(lambda v: (v>0).mean()),
                              d_tail=g.d_tail.mean())).iterrows():
    w(f'| `{c}` | {r.n_cap:.0f} | {r.d_mAP:+.4f} | {r.thang:.0%} | {r.d_tail:+.4f} |')
w(''); w('Mốc `17` (cohort cũ, 3 checkpoint cùng lần chạy): macro `+0.00112` (CI loại 0), '
         'tail `+0.00465` (CI chứa 0).'); w('')
w('## Những gì kết quả này KHÔNG trả lời'); w('')
w('- **Không có biên độ seed cho chính phép ensemble.** Đọc **tỉ lệ thắng** (bao nhiêu cặp '
  '(lần chạy, arm, seed) cùng dấu), đừng đọc độ lớn.')
w('- **Các epoch KHÔNG độc lập** — cùng quỹ đạo, tương quan rất cao. Đây là trường hợp *tệ nhất* '
  'cho ensemble; đa dạng kiến trúc (`ConvNeXtV2`) phải cho nhiều hơn, nhưng chưa đo.')
w('- **`35 §6` bác ensemble trọng số đều** khi thành viên **lệch chất lượng** (`cls` vs `patch`). '
  'Ở đây các epoch ngang nhau nên chế độ hỏng đó không áp — nhưng `all` gộp cả epoch đầu (yếu) '
  'thì có thể dính lại.')
TXT = '\n'.join(L)
ensure_res(); open(os.path.join(RES48, 'REVIEW_ENSEMBLE.md'), 'w', encoding='utf-8').write(TXT)
print(TXT)

# REVIEW · ensemble epoch & seed (`nb48`, hậu kỳ, không huấn luyện lại)

Dự đoán val đã lưu của `38c`/`38d`/`44b`/`45`. Quy tắc **cố định** (`last-k`, `all`, `seed-mean`) — không chọn epoch bằng val. `base` = **epoch cuối**, đúng giao thức `nb35 §0.1`.

| cách | #cặp | Δ mAP | tỉ lệ thắng | Δ tail |
|---|---|---|---|---|
| `all prob` | 70 | +0.0103 | 89% | +0.0086 |
| `all rank` | 70 | +0.0052 | 77% | +0.0016 |
| `last-2 prob` | 70 | +0.0003 | 83% | +0.0002 |
| `last-2 rank` | 70 | +0.0001 | 59% | -0.0002 |
| `last-3 prob` | 70 | +0.0014 | 91% | +0.0011 |
| `last-3 rank` | 70 | +0.0013 | 89% | +0.0008 |
| `last-5 prob` | 70 | +0.0040 | 89% | +0.0041 |
| `last-5 rank` | 70 | +0.0038 | 89% | +0.0037 |

Mốc `17` (cohort cũ, 3 checkpoint cùng lần chạy): macro `+0.00112` (CI loại 0), tail `+0.00465` (CI chứa 0).

## Những gì kết quả này KHÔNG trả lời

- **Không có biên độ seed cho chính phép ensemble.** Đọc **tỉ lệ thắng** (bao nhiêu cặp (lần chạy, arm, seed) cùng dấu), đừng đọc độ lớn.
- *